In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv

# 프로젝트 루트는 .env 파일이 있는 폴더 (어디서 실행해도 안전)
ENV_PATH = find_dotenv()
PROJECT_ROOT = Path(ENV_PATH).parent
load_dotenv(ENV_PATH)
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, ".env 파일에 OPENAI_API_KEY가 없습니다."

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 1. 아까 저장해둔 거대한 텍스트 파일 불러오기
txt_path = str(PROJECT_ROOT / "data" / "stat501_full_data.txt")
with open(txt_path, "r", encoding="utf-8") as f:
    full_text = f.read()

print("📂 텍스트 파일 로드 완료! 데이터 청킹(Chunking) 시작...")

# 2. 도마 세팅 (그래프와 표가 잘리지 않도록 1500자로 큼직하게 썹니다)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)

chunks = text_splitter.split_text(full_text)
print(f"총 {len(chunks)}개의 거대한 텍스트 조각으로 썰렸습니다!")

# 3. 임베딩 및 FAISS DB 생성 (데이터가 많아 1~2분 정도 소요될 수 있습니다)
print("🧠 텍스트를 벡터로 변환하여 DB를 굽고 있습니다. 잠시만 기다려주세요...")
embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)
vector_db = FAISS.from_texts(chunks, embeddings)

# 4. 완성된 DB를 내 컴퓨터 폴더에 영구 저장!
db_save_path = str(PROJECT_ROOT / "vector_db" / "faiss_stat501_db")
vector_db.save_local(db_save_path)

print(f"💾 FAISS 벡터 데이터베이스가 [{db_save_path}]에 완벽하게 저장되었습니다!")

### ℹ️ 이전 버전의 통합 코드 (보존용)

아래는 501·504 통합 작업의 초기 버전입니다.
현재는 `notebooks/build_db/build_integrated_db.ipynb`로 이전되어 더 깨끗한 방식(`add_texts()`)으로 작동합니다.
**실행하지 마세요** — 경로와 변수 참조가 옛 상태입니다.

```python
# 1. 기존 501 DB 불러오기
main_db = FAISS.load_local(r"C:/faiss_stat501_db", embeddings, allow_dangerous_deserialization=True)

# 2. 새로운 504 데이터를 위한 임시 DB 생성 (504 텍스트 조각들 사용)
db_504 = FAISS.from_texts(chunks_504, embeddings)

# 3. 504 DB 내용을 501 DB로 병합! (가장 핵심 줄)
main_db.merge_from(db_504)

# 4. 통합된 DB 저장 (이제 501+504가 하나가 됨)
main_db.save_local(r"C:/faiss_stat_integrated_db")
```
